# Build a ClawBio Skill

Create a complete skill from scratch: SKILL.md contract, Python implementation, demo data, and validation.

| | |
|---|---|
| **Level** | Intermediate |
| **Time** | ~30 min |
| **Requirements** | Google account |
| **Full tutorial** | [docs.clawbio.ai/tutorials/build-a-skill](https://docs.clawbio.ai/tutorials/build-a-skill/) |

## When to Build a Skill

Build a skill when:

1. **You do something more than three times.** If you have run the same analysis three times, it is time to encode it.
2. **You keep pasting the same instructions.** That frustration is a signal. Capture the instructions once, reuse forever.
3. **You need consistent output.** A skill guarantees the same structure every run, across users and models.
4. **You want to standardise across a team.** Skills encode best practice so every collaborator gets the same methodology.
5. **You want to unlock work you never had bandwidth for.** Skills let agents handle analysis you could never justify doing manually.

### Key Principles

- **One skill, one task.** If your skill does two unrelated jobs, split it.
- **Build before you browse.** Marketplace skills are useful as templates, but building your own teaches you the format and gives you exact control.
- **Security warning:** Third-party skills are code. They can execute scripts with your agent's permissions. Read them carefully before installing, just as you would any software package.

## Step 0: Setup

In [ ]:
!git clone --depth 1 https://github.com/ClawBio/ClawBio.git
%cd ClawBio
!pip install -q -r requirements.txt pyyaml
print("\n\u2713 ClawBio installed successfully")

## Step 1: Create the Skill Directory

Skills are **folders**, not single files. The folder bundles everything the agent needs: instructions, scripts, demo data, examples, and tests.

In [ ]:
!mkdir -p skills/my-awesome-skill/tests skills/my-awesome-skill/examples
print("\u2713 Directory created: skills/my-awesome-skill/")

## Step 2: Write the SKILL.md

The SKILL.md is the **contract**. It tells both humans and AI agents what the skill does, when to fire it, what can go wrong, and what the output looks like.

### Anatomy of an Effective Skill

The most important sections, in order of impact:

1. **Trigger** - The single most important section. If the trigger is weak or vague, the agent will never discover or fire the skill. Be loud, explicit, and list every plausible phrasing.
2. **Gotchas** - The highest-signal content. Documents where models typically go wrong. Every failure you find during testing belongs here.
3. **Workflow** - Numbered steps, not prose. Models follow structured instructions dramatically better.
4. **Example Output** - Show, do not describe. Include actual rendered samples.
5. **Safety / Agent Boundary** - What the agent must never do.

### Freedom Levels

Not all steps need the same rigidity:
- **Fragile tasks** (database queries, variant annotation, clinical thresholds): be prescriptive with exact steps.
- **Creative tasks** (report writing, strategy, literature synthesis): give guidance but leave room for reasoning.

In [ ]:
%%writefile skills/my-awesome-skill/SKILL.md
---
name: my-awesome-skill
version: 0.1.0
author: Your Name <you@example.com>
domain: genomics
description: Analyse a set of variants and produce a summary report
license: MIT

inputs:
  - name: input_file
    type: file
    format: [vcf, csv, tsv, txt]
    description: Input data file
    required: true

outputs:
  - name: report
    type: file
    format: md
    description: Analysis report

dependencies:
  python: ">=3.10"
  packages:
    - pandas>=2.0

tags: [genomics, variants, tutorial]

demo_data:
  - path: demo_input.txt
    description: Synthetic test data with 3 variants

endpoints:
  cli: python skills/my-awesome-skill/my_skill.py --input {input_file} --output {output_dir}

metadata:
  openclaw:
    trigger_keywords:
      - analyse variants
      - variant summary
      - summarise my variants
      - variant report
---

# My Awesome Skill

You are **My Awesome Skill**, a specialised ClawBio agent for variant analysis. Your role is to load a set of genomic variants and produce a structured summary report.

## Trigger

**Fire this skill when the user says any of:**
- "analyse my variants"
- "variant summary report"
- "summarise these SNPs"
- "what variants do I have"

**Do NOT fire when:**
- The user asks about drug interactions (route to pharmgx-reporter)
- The user asks about ancestry or PCA (route to claw-ancestry-pca)

## Scope

This skill loads variants from a file, counts them, and produces a summary table. It does not annotate variants against databases or predict clinical significance.

## Domain Decisions

- **Threshold for significance**: p < 0.05 after Bonferroni correction
- **Reference database**: ClinVar (release 2025-12)
- **Population frequency filter**: exclude variants with gnomAD AF > 0.01

## Workflow

1. **Validate**: Check input file exists and has required columns (rsid, chromosome, position, genotype)
2. **Load**: Read the file with pandas, skip comment lines starting with #
3. **Summarise**: Count variants per chromosome, list all genotypes
4. **Report**: Write `report.md` with variant table and summary stats
5. **Export**: Write `summary.json` with machine-readable counts

## Example Output

```markdown
# My Awesome Skill Report

Processed **3 variants**.

| rsID | Chromosome | Genotype |
|------|-----------|----------|
| rs1234567 | 1 | AG |
| rs2345678 | 2 | CC |
| rs3456789 | 3 | TT |

*ClawBio is a research tool. Not a medical device.*
```

## Gotchas

- **Gotcha 1**: The model may try to annotate variants against ClinVar or predict pathogenicity. Do not. This skill only summarises; annotation is a separate skill (vcf-annotator).
- **Gotcha 2**: If the input has no header row, the model may guess column names. Instead, reject the file and ask the user for the correct format.
- **Gotcha 3**: Do not infer zygosity from a single genotype letter. Require the full two-allele genotype (e.g. AG, not A).

## Safety

- Never report a clinical diagnosis; always include the research-use disclaimer
- Do not extrapolate results across ancestries unless validated

## Agent Boundary

The agent (LLM) dispatches and explains. The skill (Python) executes.
The agent must NOT override thresholds or invent associations.

## Chaining Partners

- `vcf-annotator`: Feed this skill's output into vcf-annotator for ClinVar/gnomAD annotation
- `profile-report`: This skill's summary can be a section in a unified profile report

## Maintenance

- **Review cadence**: Monthly or when input format conventions change
- **Staleness signals**: New file formats from DTC providers, updated column naming conventions

## Step 3: Add Demo Data

Never commit real patient data. Generate synthetic variants that exercise your skill's logic without containing identifiable information.

In [ ]:
%%writefile skills/my-awesome-skill/demo_input.txt
# Synthetic demo data for my-awesome-skill
# This is NOT real patient data
rsid	chromosome	position	genotype
rs1234567	1	12345678	AG
rs2345678	2	23456789	CC
rs3456789	3	34567890	TT

## Step 4: Write the Python Implementation

In [ ]:
%%writefile skills/my-awesome-skill/my_skill.py
#!/usr/bin/env python3
"""My Awesome Skill: analyse variants and produce a summary report."""

import argparse
import json
from pathlib import Path

import pandas as pd


def run(input_path: Path, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_path, sep="\t", comment="#")
    print(f"\u2713 Loaded {len(df)} variants from {input_path.name}")

    # --- Your analysis logic here ---
    results = {"variants_loaded": len(df), "status": "complete"}

    report_path = output_dir / "report.md"
    report_path.write_text(
        f"# My Awesome Skill Report\n\n"
        f"Processed **{len(df)} variants**.\n\n"
        f"| rsID | Chromosome | Genotype |\n"
        f"|------|-----------|----------|\n"
        + "".join(f"| {r.rsid} | {r.chromosome} | {r.genotype} |\n" for r in df.itertuples())
        + f"\n*ClawBio is a research tool. Not a medical device.*\n"
    )
    print(f"\u2713 Report written to {report_path}")

    summary_path = output_dir / "summary.json"
    summary_path.write_text(json.dumps(results, indent=2))
    print(f"\u2713 Summary written to {summary_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="My Awesome Skill")
    parser.add_argument("--input", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    args = parser.parse_args()
    run(args.input, args.output)

## Step 5: Test It

In [ ]:
!python3 skills/my-awesome-skill/my_skill.py \
  --input skills/my-awesome-skill/demo_input.txt \
  --output /tmp/my-skill-demo

In [ ]:
from IPython.display import Markdown
Markdown(open('/tmp/my-skill-demo/report.md').read())

In [ ]:
import json
with open('/tmp/my-skill-demo/summary.json') as f:
    print(json.dumps(json.load(f), indent=2))

## Step 6: Validate Your SKILL.md

Check that the YAML frontmatter has all required fields and that the key sections are present.

In [ ]:
import yaml
from pathlib import Path

skill_path = Path('skills/my-awesome-skill/SKILL.md')
text = skill_path.read_text()
front = text.split('---')[1]
meta = yaml.safe_load(front)

# Check YAML frontmatter fields
required_yaml = ['name', 'version', 'author', 'domain', 'description', 'inputs', 'outputs']
missing_yaml = [f for f in required_yaml if f not in meta]
if missing_yaml:
    print(f'\u274c Missing YAML fields: {missing_yaml}')
else:
    print(f'\u2713 YAML frontmatter valid: {meta["name"]} v{meta["version"]}')

# Check required sections in the body
required_sections = ['## Trigger', '## Gotchas', '## Example Output', '## Safety',
                     '## Agent Boundary', '## Workflow', '## Scope']
missing_sections = [s for s in required_sections if s not in text]
if missing_sections:
    print(f'\u274c Missing sections: {missing_sections}')
else:
    print(f'\u2713 All required sections present')

# Check trigger_keywords in metadata
triggers = meta.get('metadata', {}).get('openclaw', {}).get('trigger_keywords', [])
if len(triggers) < 3:
    print(f'\u26a0 Only {len(triggers)} trigger keyword(s). Recommend at least 3 for reliable discovery.')
else:
    print(f'\u2713 {len(triggers)} trigger keywords defined')

## Step 7: Stress Test Your Skill

A skill is not ready until you can run it and use the output without iterating. If you find yourself editing the output, the skill needs fixing.

**Stress test checklist:**

1. Run with the demo data. Does the output match Example Output exactly?
2. Run with an empty file. Does it fail gracefully?
3. Run with malformed input (wrong columns, missing header). Does it reject cleanly?
4. Run with a large input (100+ rows). Does performance hold?
5. Ask an AI agent to use the skill via its trigger phrases. Does it discover and fire correctly?
6. Ask an AI agent a similar-but-wrong query. Does it correctly NOT fire this skill?

**Every failure you find becomes a Gotcha.** Go back to SKILL.md and document it.

After stress testing, your Gotchas section should have at least 3 entries.

## Step 8: Submit

Your skill folder should now look like this:

```
skills/my-awesome-skill/
\u251c\u2500\u2500 SKILL.md           # Contract with Trigger, Gotchas, Example Output
\u251c\u2500\u2500 my_skill.py        # Python implementation
\u251c\u2500\u2500 demo_input.txt     # Synthetic test data
\u251c\u2500\u2500 tests/             # Test suite (red/green TDD)
\u2514\u2500\u2500 examples/          # Input/output examples
```

To submit your skill to ClawBio, fork the repository on GitHub, add your skill directory, and open a pull request.

The PR will be audited for SKILL.md conformance (see the [SKILL.md Conformance Checklist](#skill-md-conformance) below).

---

## SKILL.md Conformance Checklist {#skill-md-conformance}

Every SKILL.md in a PR is checked against this checklist:

| Check | Requirement |
|-------|------------|
| YAML: `name` | Present and matches folder name |
| YAML: `version` | Present, semver format |
| YAML: `author` | Present |
| YAML: `description` | Present, one line, specific |
| YAML: `inputs` | Present with format and required flag |
| YAML: `outputs` | Present with format |
| YAML: `trigger_keywords` | At least 3 keywords |
| Section: `## Trigger` | Present with fire/do-not-fire lists |
| Section: `## Scope` | Present, confirms one-skill-one-task |
| Section: `## Workflow` | Present, numbered steps (not prose) |
| Section: `## Example Output` | Present with actual rendered sample |
| Section: `## Gotchas` | Present with at least 3 entries |
| Section: `## Safety` | Present with disclaimer reference |
| Section: `## Agent Boundary` | Present |
| File: demo data | At least one demo file in the skill folder |
| File: tests/ | Test directory with at least one test file |
| Line count | SKILL.md under 500 lines |

---

## Skill Maintenance

Skills decay. Models change, databases update, user needs evolve.

- **Re-evaluate monthly** or when you upgrade the model version
- **Re-evaluate on upstream changes**: new ClinVar release, new gnomAD version, API endpoint changes
- **Deprecate** skills that no longer serve users. Move to `skills/_deprecated/` with a note.
- **The test**: if you have to iterate on the output after running the skill, the skill needs fixing.

---

**Previous:** [Run Your First Skill](https://docs.clawbio.ai/tutorials/run-your-first-skill/) | **Next step:** [Variant Interpretation Workshop](https://docs.clawbio.ai/tutorials/variant-interpretation-workshop/)

**Full workshop sequence:** [Run Your First Skill](https://docs.clawbio.ai/tutorials/run-your-first-skill/) > Build a Skill > [Variant Interpretation](https://docs.clawbio.ai/tutorials/variant-interpretation-workshop/) > [GWAS](https://docs.clawbio.ai/tutorials/gwas-workshop/) > [30x WGS](https://docs.clawbio.ai/tutorials/30x-wgs-workshop/)

*ClawBio is a research and educational tool. It is not a medical device.*